In [1]:
#═══════════════════════════════════════════════════════════
# One-time rebuild: Create chunked collection with timestamps
# ═══════════════════════════════════════════════════════════

import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.data_loader import load_episodes
from src.vector_store_chunked import build_chunked_collection

In [2]:
# ═══════════════════════════════════════════════════════════
# Load episode data
# ═══════════════════════════════════════════════════════════

print("Loading episodes...")
df = load_episodes()

print(f"\nDataset info:")
print(f"  Total episodes: {len(df)}")
print(f"  Columns: {df.columns.tolist()}")

# Verify raw_segments column exists
if 'raw_segments' not in df.columns:
    raise ValueError("raw_segments column not found! Check your CSV.")

print(f"\n✓ raw_segments column found")

Loading episodes...
✓ embedding type: <class 'list'> <class 'float'>
✓ Loaded 46 episodes

Dataset info:
  Total episodes: 46
  Columns: ['video_id', 'transcript_text', 'duration_mins', 'num_segments', 'raw_segments', 'status', 'url', 'transcript_clean', 'word_count', 'youtube_title', 'youtube_channel', 'error', 'fetched_at', 'embedding']

✓ raw_segments column found


In [3]:

# ═══════════════════════════════════════════════════════════
# Build chunked collection
# ═══════════════════════════════════════════════════════════

print("\nBuilding chunked collection...")
print("This will:")
print("  1. Split each episode into ~500-token chunks")
print("  2. Map each chunk to precise timestamps")
print("  3. Generate embeddings for all chunks")
print("  4. Store in ChromaDB")
print("\nEstimated time: ~10-15 minutes for 46 episodes\n")

collection = build_chunked_collection(
    df,
    collection_name="podcast_chunks",
    force_rebuild=True,          # Delete old collection
    chunk_size_tokens=500,       # ~2-3 minutes per chunk
    overlap_tokens=50,           # Context continuity
)

INFO     | ============================================================
INFO     | BUILDING CHUNKED COLLECTION
INFO     | ============================================================



Building chunked collection...
This will:
  1. Split each episode into ~500-token chunks
  2. Map each chunk to precise timestamps
  3. Generate embeddings for all chunks
  4. Store in ChromaDB

Estimated time: ~10-15 minutes for 46 episodes



⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.
INFO     | Deleted existing collection: podcast_chunks
INFO     | Created collection: podcast_chunks
INFO     | Processing 46 episodes...
INFO     | 
[1/46] Episode 001: How to Set (And Keep) Your Boundaries This Year, S...
INFO     | Episode 001: Created 138 chunks from 7602 segments
INFO     |   Created 138 chunks
INFO     | 
[2/46] Episode 002: Self-Compassion: How to Make it Work for You | Dr....
INFO     | Episode 002: Created 33 chunks from 1667 segments
INFO     |   Created 33 chunks
INFO     | 
[3/46] Episode 003: Trusting Yourself: How to Stop Self-Abandonment | ...
INFO     | Episode 003: Created 33 chunks from 1686 segments
INFO     |   Created 33 chunks
INFO     | 
[4/46] Episode 004: Stop Self-Sabotage: How to Get Out of Your Own Way...
INFO     | Episode 004: Created 36 chunks from 1873 segments
INFO     |   Created 

In [4]:

# ═══════════════════════════════════════════════════════════
# Verify collection
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

total_chunks = collection.count()
print(f"Total chunks in collection: {total_chunks}")
print(f"Average chunks per episode: {total_chunks / len(df):.1f}")

# Sample a few chunks to verify structure
sample = collection.get(limit=3, include=['metadatas', 'documents'])

print(f"\nSample chunks:")
for i in range(min(3, len(sample['ids']))):
    chunk_id = sample['ids'][i]
    metadata = sample['metadatas'][i]
    doc_preview = sample['documents'][i][:100]
    
    print(f"\n{i+1}. Chunk ID: {chunk_id}")
    print(f"   Episode: {metadata['episode_title'][:50]}...")
    print(f"   Timestamp: {metadata['start_time_display']} - {metadata['end_time_display']}")
    print(f"   Duration: {metadata['duration_display']}")
    print(f"   Tokens: {metadata['token_count']}")
    print(f"   Text preview: {doc_preview}...")

print("\n" + "="*60)
print("✅ REBUILD COMPLETE")
print("="*60)
print(f"\nYou now have {total_chunks} searchable chunks with timestamps!")
print(f"Next: Update your RAG pipeline to use 'podcast_chunks' collection")


VERIFICATION
Total chunks in collection: 1156
Average chunks per episode: 25.1

Sample chunks:

1. Chunk ID: 001_c000
   Episode: How to Set (And Keep) Your Boundaries This Year, S...
   Timestamp: 0:00 - 2:08
   Duration: 2m 8s
   Tokens: 538
   Text preview: I'm Mark Manson, number one New York Times bestselling author, and this is Drew Bernie, my intrepid ...

2. Chunk ID: 001_c001
   Episode: How to Set (And Keep) Your Boundaries This Year, S...
   Timestamp: 1:55 - 3:57
   Duration: 2m 2s
   Tokens: 545
   Text preview: confusion going on here. And so, let's let's just start out like what are boundaries? >> Yeah. >> I ...

3. Chunk ID: 001_c002
   Episode: How to Set (And Keep) Your Boundaries This Year, S...
   Timestamp: 3:47 - 5:38
   Duration: 1m 51s
   Tokens: 534
   Text preview: So, all those domains I just mentioned, >> love it >> that you can have that. Okay. So, that's kind ...

✅ REBUILD COMPLETE

You now have 1156 searchable chunks with timestamps!
Next: Update your R

In [5]:
from src.vector_store_chunked import get_chunked_collection

# Load the collection
collection = get_chunked_collection("podcast_chunks")

print(f"Total chunks: {collection.count()}")

# Sample a chunk
sample = collection.get(limit=1, include=['metadatas', 'documents'])

chunk_meta = sample['metadatas'][0]
print(f"\nSample chunk:")
print(f"  ID: {sample['ids'][0]}")
print(f"  Episode: {chunk_meta['episode_title']}")
print(f"  Time: {chunk_meta['start_time_display']} - {chunk_meta['end_time_display']}")
print(f"  Duration: {chunk_meta['duration_display']}")
print(f"  Text: {sample['documents'][0][:150]}...")

INFO     | Loaded collection: podcast_chunks (1156 chunks)


Total chunks: 1156

Sample chunk:
  ID: 001_c000
  Episode: How to Set (And Keep) Your Boundaries This Year, Solved
  Time: 0:00 - 2:08
  Duration: 2m 8s
  Text: I'm Mark Manson, number one New York Times bestselling author, and this is Drew Bernie, my intrepid co-host, lead researcher and the boundary Zen mast...
